In [5]:
# Maximum Coverage — Sensor Placement (Alias-Based Model)

from optilab.aliases import Model, GRB, quicksum
import numpy as np

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()
print("Backend:", m.backend_name)

# -------------------------------------------------------------------------------
# Problem Data (synthetic but structured)
# -------------------------------------------------------------------------------
rng = np.random.default_rng(45)

n_locations = rng.integers(10, 20)   # candidate sensor sites
n_areas = rng.integers(50, 150)       # vulnerable areas

# Each location covers a random subset of areas
# coverage[i][j] = 1 if location i covers area j
coverage = rng.integers(0, 2, size=(n_locations, n_areas))

# Ensure every area is coverable (feasibility by design)
# -> each area must be covered by at least one location
for j in range(n_areas):
    if coverage[:, j].sum() == 0:
        i = rng.integers(0, n_locations)
        coverage[i, j] = 1

k = int(0.25 * n_locations)  # number of sensors allowed

print("locations:", n_locations)
print("areas:", n_areas)
print("k:", k)

# -------------------------------------------------------------------------------
# Decision Variables
# -------------------------------------------------------------------------------
x = [m.add_binary_var(name=f"x_{i}") for i in range(n_locations)]
y = [m.add_binary_var(name=f"y_{j}") for j in range(n_areas)]  # covered or not

# -------------------------------------------------------------------------------
# Constraints
# -------------------------------------------------------------------------------

# At most k sensor locations
m.add_constraint(
    quicksum(x[i] for i in range(n_locations)) <= k
)

# Coverage logic:
# y[j] = 1 if at least one selected location covers area j
for j in range(n_areas):
    m.add_constraint(
        y[j] <= quicksum(coverage[i][j] * x[i] for i in range(n_locations))
    )

# -------------------------------------------------------------------------------
# Objective: maximize covered areas
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(y[j] for j in range(n_areas)),
    GRB.MAXIMIZE
)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract Solution
# -------------------------------------------------------------------------------
x_sol = np.array([v.x for v in x])
y_sol = np.array([v.x for v in y])

covered = int(y_sol.sum())
selected = np.where(x_sol > 0.5)[0]

print("\nSelected locations:", selected)
print("Total sensors used:", len(selected))
print("Areas covered:", covered, "/", n_areas)

Backend: mip
locations: 19
areas: 107
k: 4

Selected locations: [ 5  9 12 16]
Total sensors used: 4
Areas covered: 106 / 107
